<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_1_Confusion_Matrix_Basic_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Confusion Matrix & Basic Classification Metrics

**Author:** Brad Sheese

---

## Introduction

In this notebook, we explore how to evaluate classification models using the **confusion matrix** and key metrics: Accuracy, Precision, Recall, and F1-Score.

**Why this matters**: With our imbalanced dataset (~30% defaults), accuracy alone is misleading. A model that simply predicts "no default" for everyone would achieve ~70% accuracy without learning anything useful!

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build and interpret a confusion matrix
2. Understand why accuracy fails on imbalanced data
3. Calculate and interpret Precision, Recall, and F1-Score
4. Choose the right metric for your problem

## Setup and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load Credit Default dataset
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

# Prepare features and target (bad = 1 = default)
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

# Scale and train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Setup complete!")
print(f"Test set: {len(y_test)} samples")
print(f"Actual defaults: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

## The Confusion Matrix

The confusion matrix breaks down all predictions into four categories:

| | Predicted Negative | Predicted Positive |
|--|--|--|
| **Actually Negative** | TN (True Negative) | FP (False Positive) |
| **Actually Positive** | FN (False Negative) | TP (True Positive) |

- **TP**: Correctly predicted defaults
- **TN**: Correctly predicted non-defaults
- **FP**: Predicted default but actually good (false alarm)
- **FN**: Predicted good but actually defaulted (missed case)

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nBreakdown:")
print(f"  True Positives (TP):  {tp} - Correctly predicted defaults")
print(f"  True Negatives (TN):  {tn} - Correctly predicted non-defaults")
print(f"  False Positives (FP): {fp} - False alarms (predicted bad, was good)")
print(f"  False Negatives (FN): {fn} - Missed cases (predicted good, was bad)")

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Good (0)', 'Bad (1)'])
ax.set_yticklabels(['Good (0)', 'Bad (1)'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold')

plt.colorbar(im)
plt.tight_layout()
plt.show()

## Why Accuracy Fails on Imbalanced Data

**Accuracy** = (TP + TN) / Total = (correct predictions) / (all predictions)

On balanced data (50/50), this works well. But with 70% non-defaults, a "dumb" model that always predicts "no default" achieves 70% accuracy without learning anything!

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Naive baseline: always predict majority class
baseline_accuracy = (y_test == 0).mean()

print(f"Model Accuracy: {accuracy*100:.1f}%")
print(f"Baseline (always predict 'good'): {baseline_accuracy*100:.1f}%")
print(f"\nOur model only gains {(accuracy - baseline_accuracy)*100:.1f}% over naive!")

# Visualize
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Model', 'Naive Baseline'], [accuracy*100, baseline_accuracy*100], 
               color=['steelblue', 'gray'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Comparison')
ax.set_ylim(0, 100)
for bar, val in zip(bars, [accuracy*100, baseline_accuracy*100]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Precision: Minimize False Alarms

**Precision** = TP / (TP + FP) = "Of all positive predictions, how many were correct?"

High precision = when we say "default", we're usually right = few false alarms

**Use when:** False positives are costly (e.g., spam filtering - don't block important emails)

In [ ]:
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision*100:.1f}%")
print(f"\nInterpretation:")
print(f"  Out of {tp + fp} predictions of 'default', {tp} were actually defaults.")
print(f"  We have {fp} false alarms - predicting default when it's not.")

# Higher threshold = fewer positive predictions = potentially higher precision
y_pred_conservative = (y_proba >= 0.7).astype(int)
precision_high = precision_score(y_test, y_pred_conservative)
print(f"\nWith higher threshold (0.7):")
print(f"  Precision: {precision_high*100:.1f}%")
print(f"  But we only flag {y_pred_conservative.sum()} cases instead of {y_pred.sum()}")

## Recall: Minimize Missed Cases

**Recall** = TP / (TP + FN) = "Of all actual positives, how many did we catch?"

Also called **Sensitivity** or **True Positive Rate (TPR)**

High recall = we catch most defaults = few missed cases

**Use when:** False negatives are costly (e.g., medical diagnosis - missing cancer is dangerous)

In [ ]:
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall*100:.1f}%")
print(f"\nInterpretation:")
print(f"  Out of {tp + fn} actual defaults, we caught {tp}.")
print(f"  We MISSED {fn} default cases!")

# Lower threshold = more positive predictions = higher recall
y_pred_sensitive = (y_proba >= 0.3).astype(int)
recall_sensitive = recall_score(y_test, y_pred_sensitive)
extra_false_alarms = ((y_pred_sensitive == 1) & (y_test == 0)).sum()
print(f"\nWith lower threshold (0.3):")
print(f"  Recall: {recall_sensitive*100:.1f}%")
print(f"  But we have {extra_false_alarms} extra false alarms!")

## F1-Score: Balancing Precision and Recall

**F1** = 2 * (Precision * Recall) / (Precision + Recall)

The **harmonic mean** of precision and recall. Unlike arithmetic mean, harmonic mean penalizes imbalance:
- If precision=1.0 and recall=0.0 → F1=0.0 (not 0.5!)
- Only achieves high F1 when BOTH are high

**Use when:** You need balance between false alarms and missed cases (e.g., fraud detection)

In [ ]:
f1 = f1_score(y_test, y_pred)

print(f"F1-Score: {f1*100:.1f}%")
print(f"\nAll metrics at threshold=0.5:")
print(f"  Precision: {precision*100:.1f}%")
print(f"  Recall:    {recall*100:.1f}%")
print(f"  F1-Score:  {f1*100:.1f}%")

# Visualize all metrics
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Precision', 'Recall', 'F1-Score']
values = [precision*100, recall*100, f1*100]
colors = ['#2ecc71', '#3498db', '#9b59b6']
bars = ax.bar(metrics, values, color=colors)
ax.set_ylabel('Score (%)')
ax.set_title('Classification Metrics at Threshold=0.5')
ax.set_ylim(0, 80)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Probability Distribution and Threshold Impact

The confusion matrix depends on the **threshold** (default=0.5). Let's visualize how probabilities distribute for each actual class:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Overlapping histograms by actual class
axes[0].hist(y_proba[y_test==0], bins=35, alpha=0.7, label='Actual: Good (0)', color='green')
axes[0].hist(y_proba[y_test==1], bins=35, alpha=0.7, label='Actual: Bad (1)', color='red')
axes[0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].set_xlabel('Predicted Probability of Default')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability Distribution by Actual Class')
axes[0].legend()

# Right: Metrics across thresholds
thresholds = np.arange(0.1, 0.95, 0.05)
precisions = []
recalls = []
f1s = []
for t in thresholds:
    y_p = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_p, zero_division=0))
    recalls.append(recall_score(y_test, y_p, zero_division=0))
    f1s.append(f1_score(y_test, y_p, zero_division=0))

axes[1].plot(thresholds, precisions, 'g-', label='Precision', linewidth=2)
axes[1].plot(thresholds, recalls, 'r-', label='Recall', linewidth=2)
axes[1].plot(thresholds, f1s, 'b-', label='F1-Score', linewidth=2)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Metrics vs. Decision Threshold')
axes[1].legend()
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## When to Use Which Metric?

| Scenario | Best Metric | Why |
|----------|-------------|-----|
| Spam detection | **Precision** | False positives (good emails blocked) annoy users |
| Medical diagnosis | **Recall** | Missing a disease can be fatal |
| Fraud detection | **F1** | Balance between catching fraud and avoiding false alarms |
| Balanced classes | **Accuracy** | Works well when classes are ~50/50 |

## Key Takeaways

1. **Accuracy fails on imbalanced data** - always check class distribution
2. **Precision** = "Of positive predictions, how many were right?" → minimize false alarms
3. **Recall** = "Of actual positives, how many did we catch?" → minimize missed cases
4. **F1-Score** = harmonic mean of precision and recall → balances both
5. **Threshold matters** - adjusting it trades off precision vs. recall